# 06 — ProbLog Probabilistic Reasoning

Demonstrates the ProbLog adapter via `Store.evaluate(mode="problog")`:
- Schema + facts with confidence via `sdk.batch()`
- `Derivation` with `ProbLogRuleExt(branch_probabilities=...)`
- `sdk.evaluate(engine_options={"timeout": ...})`
- `sdk.accept()` + `persist_problog_annotations`

**Requires:** `problog` CLI (`pip install problog`). Graceful fallback if not installed.  
**Prerequisites:** [01](01_sdk_basics.ipynb)–[02](02_rules_and_derivations.ipynb).  
**Next:** [07_evidence_graph_multi_engine.ipynb](07_evidence_graph_multi_engine.ipynb)

## 0. Imports

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve().parent / "src"))

In [2]:
from __future__ import annotations
from pprint import pprint

from factpy_kernel.sdk import SDKStore, Entity, Identity, Field
from factpy_kernel.sdk.dsl import Derivation, Pred, vars as sdk_vars

import factpy_kernel.adapters.problog  # registers engine evaluator
from factpy_kernel.adapters.problog.rule_ext import ProbLogRuleExt
from factpy_kernel.adapters.problog.accept import persist_problog_annotations

## 1. Schema & Seed Facts

Write facts with confidence — ProbLog uses these as observation probabilities.

In [3]:
class Researcher(Entity):
    researcher_id: str = Identity(primary_key=True)
    name: str = Field(cardinality="single")
    expertise: str = Field(cardinality="single")
    impact_score: str = Field(cardinality="single")

sdk = SDKStore([Researcher])
seeded_researchers = ["Alice", "Bob"]

# Write facts using SDK batch API (not internal set_field)
with sdk.batch(meta={"source": "faculty_db"}) as tx:
    alice = tx.entity(Researcher, researcher_id="Alice")
    alice.name.set("Alice Chen", meta={"confidence": 1.0})
    alice.expertise.set("NLP", meta={"confidence": 0.85})

    bob = tx.entity(Researcher, researcher_id="Bob")
    bob.name.set("Bob Zhang", meta={"confidence": 1.0})
    bob.expertise.set("CV", meta={"confidence": 0.85})
    tx.commit()

print({"seed_count": len(seeded_researchers), "researcher_ids": seeded_researchers})


{'seed_count': 2, 'researcher_ids': ['Alice', 'Bob']}


## 2. Define ProbLog Derivation

`ProbLogRuleExt(branch_probabilities=...)` specifies the probability of each
disjunctive branch in the ProbLog program.

In [4]:
with sdk_vars("r", "name") as (r, name):
    prob_derivation = Derivation(
        id="drv.expertise_discovery",
        version="v1",
        where=[Pred("researcher:name", r, name)],
        target="researcher:expertise",
        head_vars=[r, name],
        mode="problog",
        engine_ext=ProbLogRuleExt(branch_probabilities=(0.85,)),
    )

print(f"Derivation: mode={prob_derivation.mode}")
print(f"  engine_ext: {prob_derivation.engine_ext}")
print(f"  branch_probabilities: {prob_derivation.engine_ext.branch_probabilities}")

Derivation: mode=problog
  engine_ext: ProbLogRuleExt(branch_probabilities=(0.85,))
  branch_probabilities: (0.85,)


## 3. Evaluate

`sdk.evaluate()` dispatches to the ProbLog adapter, which:
1. Exports schema + facts to ProbLog syntax
2. Runs ProbLog CLI with `--trace`
3. Parses output probabilities and proof traces
4. Returns `CandidateSet` with provenance

If the ProbLog CLI is unavailable, the next cell prints the actual runtime exception and leaves `candidates` empty.


In [5]:
_PROBLOG_OK = False
try:
    candidates = sdk.evaluate(
        prob_derivation,
        engine_options={"timeout": 10},
    )
    _PROBLOG_OK = True
    print(f"ProbLog candidates: {len(candidates)}")
    for c in candidates:
        print(f"  target={c.target}, confidence={c.confidence}, kind={c.confidence_kind}")
        print(f"  support_kind={c.support_kind}")
except Exception as exc:
    candidates = []
    print(f"ProbLog unavailable: {type(exc).__name__}: {exc}")


ProbLog unavailable: ProbLogEngineError: ProbLog CLI is not available: problog


## 4. Accept + Persist Annotations

Use `sdk.accept()` (public API) to persist candidates.
If evaluation produced no candidates, the next cell prints a skip message derived from runtime state.


In [6]:
if _PROBLOG_OK and candidates:
    for cand in candidates:
        result = sdk.accept(cand, approved_by="demo", note="problog demo")
        print(f"Accepted: {result}")

    # Persist ProbLog-specific annotations
    persist_problog_annotations(sdk.ledger, candidates)
    print(f"Persisted ProbLog annotations for {len(candidates)} candidate(s)")
else:
    print(f"Acceptance skipped: {len(candidates)} candidate(s) available")


Acceptance skipped: 0 candidate(s) available


## 5. Architecture Notes

```
Derivation(mode='problog', engine_ext=ProbLogRuleExt(...))
    │
    ▼
Store.evaluate()  →  evaluate_problog()
    │                    │
    │                    ├── export_problog()  →  ProbLog program text
    │                    ├── run_problog()     →  CLI with --trace
    │                    └── parse output      →  CandidateSet + ProbLogTraceV0
    │
    ▼
CandidateSet
  .confidence  = probability from ProbLog output
  .support_kind = 'problog_provenance_v1'
  .support_artifact = ProbLogTraceV0 (proof trace)
```

**Explain surface status:**

| Surface | ProbLog 状态 | 原因 |
|---------|-------------|------|
| `explain-tree` (CandidateEvidenceTree) | ✅ 已支持（accepted candidate） | runtime 会把 `ProbLogTraceV0` 投影成 CandidateEvidenceTree；pre-accept 或 payload 不可回溯时返回 `explain_not_supported` |
| `explain-timeline` (CandidateProvenanceTimeline) | ❌ 不适用 | ProbLog trace 是 proof tree 形态，不是 event log |
| `explain-summary` / `explain-narrative` / `explain-nl` | ✅ 已支持（accepted candidate） | 复用同一 tree contract；summary 追加 `problog_probability`，narrative/NL 继续传递 probability |
| `EvidenceGraph(tree)` in audit package | ✅ 可用 | ProbLogTraceV0 → problog_trace_to_evidence_graph() |

ProbLog 的 runtime explain 现在已经接入 `CandidateEvidenceTree`，但边界是刻意收敛的：
- `ProbLogTraceV0` 本身是 tree 形的 proof trace（call frame tree），因此沿用 `CandidateEvidenceTree`，不新建 timeline 合同
- raw `explain(kind="candidate")` 仍返回 engine-native provenance envelope；tree family 是在 runtime 上层的投影 surface
- projected tree 使用 `proof_goal` / `proof_leaf`，不复用 witness 语义，也不会生成 synthetic assertion 链接
- candidate anchoring 仍是 best-effort，但当前已足够冻结为 accepted-candidate 的 runtime tree contract
- 这和 PyReason 不同——PyReason 的原始载体是 event log，所以需要独立的 `CandidateProvenanceTimeline`

**Export priority chain:**
`problog/semantic/probability` → `shared/semantic/probability` → `meta.confidence` → 1.0

**Key design principle:** probability is a fact semantic property, not an engine parameter.

---
**Next:** [07_evidence_graph_multi_engine.ipynb](07_evidence_graph_multi_engine.ipynb)